# 08 — AgentDojo Benchmark (SPQ vs baselines)

Runs SPQ against the **AgentDojo** tool-use IPI benchmark.
Reports: injected task success rate (↓ = better defense) and benign PSR (↑ = utility preserved).

Corresponds to **Table T1** in the thesis.

Full runner: `scripts/run_agentdojo_suite.py`

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
BACKEND      = "openrouter"          # "openrouter" | "ollama"
MODEL        = "openai/gpt-4o-mini"
JUDGE_MODEL  = "anthropic/claude-haiku-4-5"  # must differ from victim
N_TASKS      = 10                    # subset for demo (full=97)
DEFENSES     = ['none', 'spq']       # add 'smoothllm', 'struq' for full comparison
SPQ_K        = 3
SPQ_Q        = 2
SEED         = 42
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import sys, os, json, random
sys.path.insert(0, os.path.abspath('..'))
from dotenv import load_dotenv
load_dotenv(dotenv_path=os.path.join('..', '.env'), override=False)
load_dotenv(dotenv_path=os.path.expanduser('~/.env'), override=False)
random.seed(SEED)

In [ ]:
# Check AgentDojo availability
try:
    import agentdojo
    print(f'AgentDojo version: {agentdojo.__version__}')
    USE_NATIVE = True
except ImportError:
    print('AgentDojo not installed. Using bundled in-house tasks.')
    print('Install with: pip install agentdojo')
    USE_NATIVE = False

In [ ]:
from aaps.defenses.pace.pipeline import PACEDefense

def build_defense(name):
    if name == 'none':
        return None
    if name == 'spq':
        return PACEDefense(config={'model': MODEL, 'K': SPQ_K, 'q': SPQ_Q, 'backend': BACKEND})
    # Add other baselines here
    raise ValueError(f'Unknown defense: {name}')

print('Defense factory ready.')

In [ ]:
if USE_NATIVE:
    from aaps.evaluation.agentdojo_native import run_agentdojo_suite
    results = {}
    for defense_name in DEFENSES:
        d = build_defense(defense_name)
        r = run_agentdojo_suite(
            defense=d,
            model=MODEL,
            backend=BACKEND,
            n_tasks=N_TASKS,
            seed=SEED,
        )
        results[defense_name] = r
        print(f'{defense_name}: injected_SR={r["injected_sr"]:.3f}  benign_PSR={r["psr"]:.3f}')
else:
    print('Native AgentDojo not available. Run: python scripts/run_agentdojo_suite.py')
    print('Or install agentdojo: pip install agentdojo')
    results = {}

In [ ]:
# Display results table
if results:
    import pandas as pd
    rows = []
    for name, r in results.items():
        rows.append({
            'Defense': name,
            'Injected SR (↓)': f"{r['injected_sr']:.3f}",
            'Benign PSR (↑)': f"{r['psr']:.3f}",
            'Latency p50 (ms)': f"{r.get('latency_p50_ms', '?')}",
        })
    df = pd.DataFrame(rows)
    display(df)
else:
    print('No results to display. Configure AgentDojo and re-run.')

## Full run

```bash
python scripts/run_agentdojo_suite.py \
  --backend openrouter --model openai/gpt-4o-mini \
  --defenses none spq smoothllm struq prompt_guard2 \
  --n-tasks 97 --seeds 42 43 44 \
  --out logs/thesis/$(date +%H%M-%d%m%Y)/agentdojo/
```

Results land in `logs/thesis/<timestamp>/agentdojo/` and are auto-rendered
into `Generated/tab_agentdojo_baselines.tex` by `scripts/reporting/render_thesis_tables.py`.